# Udeco 多尺度物理分量分解实验

本 Notebook 用于检查单路 Udeco 信号能否近似拆分为

$$
U(t)=B_{\mathrm{slow}}(t)+P(t)+J(t)+n(t)
$$

- $J(t)$：接触或脱离引起的快速、有限时长跃迁；
- $P(t)$：压迫增强、保持和松弛造成的中尺度变化；
- $B_{\mathrm{slow}}(t)$：汗液、温度和长期界面变化产生的超慢漂移；
- $n(t)$：上述成分之外的剩余噪声。

主线固定为 **Hampel → Multi-scale Ramp/Haar → Robust LOWESS → Conditional Bslow**。
Udeco2 与 Udeco3 使用同一套函数，但分别估计局部噪声、事件和基线。


In [ ]:
# ============================================================
# 用户配置：正常情况下只需修改 SAMPLE_NAME，然后从头运行 Notebook
# 所有时间参数均使用秒；算法内部会根据实际 fs 自动换算为采样点。
# ============================================================

SAMPLE_NAME = "ts_write_1"

# 数据根目录。增减无算法意义；仅在数据目录移动时修改。
# 推荐：保持为仓库共享 subject_data，或通过 UDECO_DATA_ROOT 环境变量覆盖。
DATA_ROOT = r"D:\MHLcode\subject_data"

# Pydisplay 协议采样率（Hz）。当前仓库数据固定为 100 Hz；通常不要修改。
# 增大/减小会按比例改变所有秒制窗口对应的采样点数；推荐仅使用真实采样率。
FS_HZ = 100

# =========================
# 预处理
# =========================
# Hampel 局部窗口（秒）。增大：局部中位数更稳但更可能把短边沿视为异常；
# 减小：更保护快速变化但对宽一点的尖峰不敏感。推荐 0.03–0.11 s。
HAMPEL_WINDOW_SEC = 0.05

# Hampel 异常阈值（局部稳健标准差倍数）。增大：只删除极端尖峰；
# 减小：删除更多点但可能误伤真实变化。推荐 3–6，第一版取 4.5。
HAMPEL_N_SIGMA = 4.5

# =========================
# J：快速接触/脱离跃迁
# =========================
# 候选过渡时长（秒）。较大尺度能发现较慢接触，较小尺度更敏感于快速边沿；
# 范围过宽会增加重复候选。100 Hz 数据推荐覆盖 0.05–0.40 s。
EDGE_SCALES_SEC = [0.05, 0.1, 0.2, 0.4]

# 过渡前稳定平台窗口（秒）。增大：平台中位数更稳但容易混入邻近事件；
# 减小：定位更局部但 MAD 波动更大。推荐 0.15–0.80 s。
EDGE_PRE_WINDOW_SEC = 0.30

# 过渡后平台/持续性窗口（秒）。增大：更严格确认新平台保持；
# 减小：允许更短平台但误检风险上升。推荐 0.20–1.00 s。
EDGE_POST_WINDOW_SEC = 0.50

# 前后平台差的局部 MAD 阈值。增大：减少误检但漏掉弱跃迁；
# 减小：检出更灵敏但噪声候选更多。推荐 4–8。
EDGE_Z_THRESHOLD = 5.0

# 接受事件的最短实际跃迁时长（秒）。增大：排除更快变化；
# 减小：更接近单点阶跃。100 Hz 下推荐 0.02–0.10 s，取 0.03 s。
EDGE_MIN_DURATION_SEC = 0.03

# 有向 Spearman 单调性阈值，范围 0–1。增大：只接受更平滑的单向变化；
# 减小：容忍抖动但可能接收无方向波动。推荐 0.50–0.85。
EDGE_MONOTONICITY_THRESHOLD = 0.65

# J 事件两侧额外屏蔽时长（秒）。增大：Bm 更不受边沿残差影响；
# 减小：保留更多有效趋势点。推荐 0.05–0.50 s。
EDGE_MASK_MARGIN_SEC = 0.15

# =========================
# Bm：中尺度趋势
# =========================
# 中尺度趋势算法。第一版固定为 robust_loess；保留该参数便于未来替换实现。
MEDIUM_METHOD = "robust_loess"

# LOWESS 等效局部窗口（秒）。增大：Bm 更平滑但可能削弱压迫缓坡；
# 减小：更贴近原信号但残留更多高频噪声。推荐 0.5–5.0 s。
MEDIUM_WINDOW_SEC = 2.0

# LOWESS 残差稳健重加权次数。增大：更抗离群点但更慢；
# 减小：计算更快但更易受异常拖动。推荐 1–4 次。
MEDIUM_ROBUST_ITERS = 2

# =========================
# Bslow：超慢基线
# =========================
# 条件 EWMA 的名义时间常数（秒）。增大：基线更慢、更不吸收压迫；
# 减小：更快跟踪长期漂移但可能侵蚀 P。推荐 30–300 s。
BSLOW_TAU_SEC = 120.0

# 名义 EWMA 更新倍率，实际 alpha 仍按 fs 和 tau 换算。增大：稳定段跟踪更快；
# 减小：更新更保守。推荐 0.1–1.0，且必须位于 (0, 1]。
BSLOW_UPDATE_RATE = 1.0

# 允许更新时的最大绝对 edge score。增大：边沿附近也可能更新；
# 减小：冻结范围更保守。推荐 1.5–4.0，通常低于事件检测阈值。
BSLOW_STABLE_EDGE_THRESHOLD = 2.5

# 允许更新时的最大 |dBm/dt|（Udeco/s）。增大：压迫缓坡中也可能更新；
# 减小：只在更平坦区更新。推荐 5e-5–1e-3，需结合调试图调整。
BSLOW_STABLE_SLOPE_THRESHOLD = 2e-4

# 允许更新时 Bm 相对当前 Bslow 的最大偏移（Udeco）。增大：高平台可能被吸收；
# 减小：基线更易长期冻结。推荐 1e-4–2e-3。
BSLOW_STABLE_OFFSET_THRESHOLD = 5e-4

# =========================
# 可视化
# =========================
# 局部观察起止时间（秒）。None 表示使用记录边界；增减只影响绘图，不影响算法。
PLOT_START_SEC = None
PLOT_END_SEC = None

# False 显示真实 Udeco 数值；True 仅对绘图副本做稳健标准化，便于比较形态。
# 该开关绝不会修改 result2/result3 中的实际分解结果。
NORMALIZE_COMPONENTS_FOR_PLOT = False


## 环境与数据读取

数据读取完全复用 `udeco_analysis.load_udeco_sample`。它负责序号修复、缺失点插值、
读取共享 `ck.mat` 标定并计算 Udeco2/Udeco3；本 Notebook 不重复实现这些仓库契约。


In [ ]:
from __future__ import annotations

import math
import os
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import statsmodels
from IPython.display import display
from scipy.stats import spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess

from udeco_analysis import load_udeco_sample

print(
    f"Python {platform.python_version()} | NumPy {np.__version__} | "
    f"SciPy {scipy.__version__} | statsmodels {statsmodels.__version__}"
)


In [ ]:
# 环境变量优先，便于换机器；第一单元的 DATA_ROOT 仍是可见且统一的默认值。
data_root = Path(os.environ.get("UDECO_DATA_ROOT", DATA_ROOT)).expanduser().resolve()
if not data_root.is_dir():
    raise NotADirectoryError(f"数据根目录不存在：{data_root}")

# SAMPLE_NAME 不含 `_sensor.csv`。只接受唯一匹配，避免静默读错受试者或动作批次。
sample_matches = sorted(data_root.glob(f"*/{SAMPLE_NAME}_sensor.csv"))
if not sample_matches:
    raise FileNotFoundError(
        f"没有找到样本 {SAMPLE_NAME!r}；期望路径形如 "
        f"{data_root}/<subject>/{SAMPLE_NAME}_sensor.csv"
    )
if len(sample_matches) > 1:
    raise RuntimeError(f"样本名不唯一：{[str(path) for path in sample_matches]}")

sensor_file = sample_matches[0]
sample = load_udeco_sample(sensor_file, data_root=data_root, fs=FS_HZ)

# 后续算法只依赖这四个变量，因此切换样本不需要改动任何函数调用。
time = sample.time_s.copy()
udeco2 = sample.udeco2.copy()
udeco3 = sample.udeco3.copy()
fs = sample.fs

if not (len(time) == len(udeco2) == len(udeco3)):
    raise RuntimeError("time、Udeco2、Udeco3 长度不一致。")
if len(time) == 0 or not (np.isfinite(udeco2).all() and np.isfinite(udeco3).all()):
    raise RuntimeError("Udeco 数据为空或包含无法恢复的非有限值。")


In [ ]:
# 只输出紧凑摘要，避免在 Notebook 中打印整段原始数组。
load_summary = pd.Series(
    {
        "sample_name": sample.sample_name,
        "sensor_file": str(sensor_file),
        "fs_hz": fs,
        "n_samples": len(time),
        "duration_sec": time[-1] if len(time) else 0.0,
        "Udeco2_min": float(np.min(udeco2)),
        "Udeco2_max": float(np.max(udeco2)),
        "Udeco3_min": float(np.min(udeco3)),
        "Udeco3_max": float(np.max(udeco3)),
        "sequence_interpolated": int(sample.qc["InterpFlag"].sum()),
    },
    name="value",
)
display(load_summary.to_frame())


## 后续分解路径

1. Hampel 仅去除孤立尖峰，保留 `raw` 与 `clean`；
2. 多尺度平台—过渡—平台检测重建累计跃迁分量 $J(t)$；
3. 在减去 J 并弱化事件区域后，用 Robust LOWESS 估计 $B_m(t)$；
4. 仅在真正稳定区更新 $B_{\mathrm{slow}}(t)$，最后得到 $P=B_m-B_{\mathrm{slow}}$。

后续阶段会逐层加入函数、关键合成检查和诊断图。
